# 🏥 Sprint 4: Redes Neuronales Convolucionales (CNN) para Visión Artificial en Triaje
## Clasificación de Radiografías de Tórax (Sano vs Neumonía)
**Proyecto Integrador de Inteligencia Artificial**  
**Autor:** Nain  

Este notebook está optimizado para ejecutarse en **Google Colab** utilizando aceleración por GPU (T4).

### 1. Importación de Librerías y Verificación de GPU

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

print(f"Versión de TensorFlow: {tf.__version__}")
print(f"Dispositivos disponibles: {tf.config.list_physical_devices('GPU')}")

### 2. Construcción de la Arquitectura CNN del Proyecto
Diseño convolucional con 3 bloques de extracción de características `Conv2D + MaxPooling2D` seguidos de aplanado, regularización con `Dropout(0.5)` y neurona de salida sigmoidal.

In [ ]:
# Construir la arquitectura de la CNN
modelo_cnn = models.Sequential([
    # Primera capa convolucional y de pooling
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    
    # Segunda capa
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    # Tercera capa
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    # Aplanado y capas densas (clasificación)
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Evita sobreajuste
    layers.Dense(1, activation='sigmoid') # 1 = Neumonía / Riesgo, 0 = Sano
])

modelo_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')]
)

modelo_cnn.summary()

### 3. Pipeline de Aumento de Datos (Data Augmentation)
Rotación, traslación y reescalado de imágenes a 150x150 para prevenir sobreajuste y simular variabilidad anatómica.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

### 4. Entrenamiento del Modelo con Callbacks (Early Stopping y Checkpoint)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ModelCheckpoint('modelo_cnn_neumonia.keras', save_best_only=True)
]

print("Modelo listo para entrenar sobre dataset de tórax.")

### 5. Curvas de Aprendizaje y Evaluación de Métricas Clínicas

In [ ]:
# Visualización de métricas tras entrenamiento
print("Evaluando recall clínico para descartar falsos negativos en neumonía...")